# Corretude de algoritmos iterativos e invariantes de laço — Parte 2 — Tutorial

**Algoritmos e Estruturas de Dados II (COMP0498) — UFS — 2026.2**

## Objetivos

Ao final deste tutorial você será capaz de:

- Provar a corretude da **busca binária** usando um invariante condicional e uma medida
  decrescente para o término;
- Reconhecer (e consertar) os dois erros clássicos da busca binária: janela que não
  encolhe e overflow no cálculo do meio;
- Provar a corretude do **particionamento de Lomuto** com o invariante das quatro regiões;
- Usar o invariante como **ferramenta de projeto**: escrever o laço a partir do invariante,
  e não o contrário.


In [ ]:
# Verifique se o gcc está disponível no seu ambiente
!gcc --version | head -1


## 1. Busca binária

Precondição: `A[0..n-1]` **ordenado**. O invariante é *condicional* — ele não promete que
`x` está na janela, promete que **fora dela é impossível**:

> No início de cada iteração: se `x` ocorre em `A[0..n-1]`, então `x` ocorre em `A[lo..hi]`.

- **Inicialização**: a janela `[0..n-1]` é o vetor inteiro. ✓
- **Manutenção**: se `A[meio] < x`, a ordenação garante que `A[k] <= A[meio] < x` para todo
  `k <= meio` — descartar `[lo..meio]` é seguro. O caso `A[meio] > x` é simétrico. ✓
- **Término**: se saímos com `lo > hi`, a janela está vazia; pelo invariante, `x` não está
  no vetor — devolver `-1` está *provado*. ✓
- **O laço termina**: o tamanho da janela `hi - lo + 1` é ≥ 0 e decresce estritamente,
  porque os dois ramos **excluem** `meio` da nova janela.

No código abaixo, o `assert` confere o invariante recalculando, por força bruta, se `x`
aparece no vetor e se aparece na janela.


In [ ]:
%%writefile busca_binaria.c
#include <stdio.h>
#include <assert.h>

static int ocorre(int A[], int a, int b, int x) {  /* x ocorre em A[a..b]? */
    for (int k = a; k <= b; k++)
        if (A[k] == x) return 1;
    return 0;
}

int busca_binaria(int A[], int n, int x) {   /* pre: A ordenado */
    int lo = 0, hi = n - 1;
    while (lo <= hi) {
        /* invariante: ocorre(A,0,n-1,x) -> ocorre(A,lo,hi,x) */
        assert(!ocorre(A, 0, n - 1, x) || ocorre(A, lo, hi, x));
        printf("janela [%d..%d]\n", lo, hi);

        int meio = lo + (hi - lo) / 2;
        if (A[meio] == x)      return meio;
        else if (A[meio] < x)  lo = meio + 1;
        else                   hi = meio - 1;
    }
    return -1;
}

int main(void) {
    int A[] = {1, 3, 4, 7, 9, 11, 15, 20};
    printf("busca 11 -> indice %d\n", busca_binaria(A, 8, 11));
    printf("busca 5  -> indice %d\n", busca_binaria(A, 8, 5));
    return 0;
}


In [ ]:
# Compila e executa: o rastro mostra a janela encolhendo a cada iteração
!gcc -Wall busca_binaria.c -o busca_binaria && ./busca_binaria
!./busca_binaria | grep -q 'busca 11 -> indice 5' && ./busca_binaria | grep -q 'busca 5  -> indice -1' && echo OK || echo 'Verifique: esperava indices 5 e -1'


### O erro que a prova de término denuncia

Troque `lo = meio + 1` por `lo = meio`. A manutenção do invariante continua válida
(a janela ainda contém `x`, se ele existir) — mas o **término** quebra: com janela
`[3..4]`, `meio = 3`; se `A[3] < x`, a "nova" janela continua `[3..4]` e o laço nunca
avança. A prova exige que a medida `hi - lo + 1` **decresça estritamente**, e é o
`+ 1` que garante isso.

A célula abaixo executa a versão errada com um contador de segurança para você ver o
laço estagnar (sem travar o notebook).


In [ ]:
%%writefile busca_binaria_errada.c
#include <stdio.h>

int busca_binaria_errada(int A[], int n, int x) {
    int lo = 0, hi = n - 1;
    int passos = 0;
    while (lo <= hi) {
        if (++passos > 20) {               /* trava de seguranca */
            printf("abortado: 20 iteracoes sem terminar (laco infinito)\n");
            return -2;
        }
        printf("janela [%d..%d]\n", lo, hi);
        int meio = lo + (hi - lo) / 2;
        if (A[meio] == x)      return meio;
        else if (A[meio] < x)  lo = meio;   /* ERRO: deveria ser meio + 1 */
        else                   hi = meio - 1;
    }
    return -1;
}

int main(void) {
    int A[] = {1, 3, 4, 7, 9, 11, 15, 20};
    printf("busca 20 -> %d\n", busca_binaria_errada(A, 8, 20));
    return 0;
}


In [ ]:
# A janela para de encolher: a medida hi - lo + 1 não decresce
!gcc -Wall busca_binaria_errada.c -o busca_binaria_errada && ./busca_binaria_errada


In [ ]:
%%writefile exercicio1.c
#include <stdio.h>

/* Exercício 1: busca binaria do LIMITE INFERIOR (lower bound).
   Devolva o MENOR indice i tal que A[i] >= x; se nao existir, devolva n.
   Invariante sugerido (janela [lo..hi), meio-aberta):
     - todo elemento de A[0..lo-1] e  < x
     - todo elemento de A[hi..n-1] e >= x
   Termino: quando lo == hi, esse indice comum e a resposta.
   TODO: implemente mantendo o invariante. Cuidado: aqui hi comeca em n,
   e o laco roda enquanto lo < hi. */
int limite_inferior(int A[], int n, int x) {
    /* TODO: implemente aqui */
    return -1;
}

int main(void) {
    int A[] = {1, 3, 3, 3, 7, 9};
    printf("lb(3) = %d\n", limite_inferior(A, 6, 3));
    printf("lb(5) = %d\n", limite_inferior(A, 6, 5));
    printf("lb(10) = %d\n", limite_inferior(A, 6, 10));
    return 0;
}


In [ ]:
# Teste automático do Exercício 1
!gcc -Wall exercicio1.c -o exercicio1 && ./exercicio1
!./exercicio1 | grep -q 'lb(3) = 1' && ./exercicio1 | grep -q 'lb(5) = 4' && ./exercicio1 | grep -q 'lb(10) = 6' && echo OK || echo 'Verifique sua implementação: esperava lb(3)=1, lb(5)=4, lb(10)=6'


## 2. Particionamento de Lomuto

O particionamento reorganiza `A[p..r]` em torno do pivô `A[r]` e é o coração do quicksort.
O invariante descreve **quatro regiões** delimitadas pelos índices `i` e `j`:

> No início da iteração `j`:
> `A[p..i] <= pivô`  |  `A[i+1..j-1] > pivô`  |  `A[j..r-1]` não examinado  |  `A[r] = pivô`.

- **Inicialização** (`i = p-1`, `j = p`): as duas primeiras regiões são vazias — as
  cláusulas valem por **vacuidade**. ✓
- **Manutenção**: se `A[j] > pivô`, só avançar `j` (a região "maiores" cresce);
  se `A[j] <= pivô`, incrementar `i` e trocar `A[i] ↔ A[j]` (a região "menores" cresce). ✓
- **Término** (`j = r`): tudo classificado; a troca final põe o pivô em `A[i+1]`,
  entre as duas regiões. ✓


In [ ]:
%%writefile particiona.c
#include <stdio.h>
#include <assert.h>

static void troca(int *a, int *b) { int t = *a; *a = *b; *b = t; }

int particiona(int A[], int p, int r) {
    int pivo = A[r];
    int i = p - 1;
    for (int j = p; j < r; j++) {
        /* invariante: A[p..i] <= pivo  e  A[i+1..j-1] > pivo */
        for (int k = p; k <= i; k++)     assert(A[k] <= pivo);
        for (int k = i + 1; k < j; k++)  assert(A[k] >  pivo);

        if (A[j] <= pivo) {
            i++;
            troca(&A[i], &A[j]);
        }
    }
    troca(&A[i + 1], &A[r]);
    return i + 1;
}

int main(void) {
    int A[] = {3, 8, 1, 5};
    int q = particiona(A, 0, 3);
    printf("pivo na posicao %d: ", q);
    for (int k = 0; k < 4; k++)
        printf("%d ", A[k]);
    printf("\n");
    return 0;
}


In [ ]:
# Compila e executa: o exemplo rastreado nos slides
!gcc -Wall particiona.c -o particiona && ./particiona
!./particiona | grep -q 'pivo na posicao 2: 3 1 5 8' && echo OK || echo 'Verifique: esperava pivo na posicao 2: 3 1 5 8'


In [ ]:
%%writefile exercicio2.c
#include <stdio.h>

/* Exercício 2: separacao pares/impares.
   Reorganize A[0..n-1] para que todos os PARES fiquem antes de todos os
   IMPARES, devolvendo o indice do primeiro impar (ou n se nao houver).
   Adapte o invariante de Lomuto (aqui nao ha pivo):
     A[0..i]   pares  |  A[i+1..j-1] impares  |  A[j..n-1] nao examinado
   TODO: implemente mantendo o invariante. */
int separa_pares(int A[], int n) {
    /* TODO: implemente aqui */
    return -1;
}

int main(void) {
    int A[] = {7, 2, 9, 4, 6, 1};
    int q = separa_pares(A, 6);
    printf("primeiro impar em %d\n", q);
    /* confere a poscondicao */
    int ok = 1;
    for (int k = 0; k < q; k++)  if (A[k] % 2 != 0) ok = 0;
    for (int k = q; k < 6; k++)  if (A[k] % 2 == 0) ok = 0;
    printf("poscondicao: %s\n", ok ? "valida" : "violada");
    return 0;
}


In [ ]:
# Teste automático do Exercício 2
!gcc -Wall exercicio2.c -o exercicio2 && ./exercicio2
!./exercicio2 | grep -q 'primeiro impar em 3' && ./exercicio2 | grep -q 'poscondicao: valida' && echo OK || echo 'Verifique sua implementação: esperava primeiro impar em 3 e poscondicao valida'


### Para pensar (responda em uma célula de texto)

1. Na busca binária, o invariante é uma **implicação** ("se `x` está no vetor, está na
   janela"). O que aconteceria com a prova se o enunciássemos como "x está em `A[lo..hi]`"
   sem o "se"? Em qual passo ela quebraria?
2. No particionamento, a inicialização vale **por vacuidade** (regiões vazias). Cite outro
   algoritmo deste tutorial ou da Parte 1 em que o caso base também é uma região vazia ou
   unitária.


## Desafio Final

Monte o **quicksort completo** e verifique sua poscondição:

1. Reaproveite a `particiona` deste tutorial (com os `assert` do invariante);
2. Escreva `void quicksort(int A[], int p, int r)` — o argumento de corretude da recursão
   é por **indução forte** no tamanho do trecho: o particionamento (provado por invariante)
   garante que cada chamada recursiva recebe um subproblema estritamente menor e que juntar
   as partes preserva a ordenação;
3. Verifique a poscondição completa (ordenado + permutação da entrada) usando o
   verificador que você construiu no Desafio Final da Parte 1;
4. Teste com: vetor já ordenado, vetor em ordem reversa, vetor com todos os elementos
   iguais e um vetor com 1 elemento.

Extra: o que acontece com o invariante de Lomuto quando todos os elementos são iguais ao
pivô? A prova continua valendo? E o desempenho?


In [ ]:
%%writefile desafio.c
#include <stdio.h>

/* Desafio: quicksort completo com verificacao de poscondicao */

static void troca(int *a, int *b) { int t = *a; *a = *b; *b = t; }

int particiona(int A[], int p, int r) {
    /* TODO: copie sua versao com asserts do invariante */
    return -1;
}

void quicksort(int A[], int p, int r) {
    /* TODO: implemente a recursao */
}

int main(void) {
    /* TODO: teste com os quatro casos pedidos e verifique a poscondicao */
    return 0;
}


In [ ]:
# Compile e teste seu desafio
!gcc -Wall desafio.c -o desafio && ./desafio


## Referências

Veja o arquivo `../referencias.bib` para a lista completa.
